# 00. Raw Data Audit

Goal: find out what is wrong with the raw Kaggle file `messi_all_goals.csv` **before** any cleaning.
This notebook reads the raw file only and does not use `src/clean.py`. It ends with a to-do list that `clean.py` implements.

**Sections**
1. Overview
2. Issues that need cleaning
3. Other observations
4. Cleaning to-do list

In [6]:
from pathlib import Path

import pandas as pd

# The notebook lives in notebooks/, so the repository root is one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = REPO_ROOT / "data" / "raw" / "messi_all_goals.csv"

## 1. Overview

The file is read with every column as text, so pandas does not guess any types.

In [17]:
raw = pd.read_csv(RAW_PATH)

print(f"Shape: {raw.shape}")
print(f"Duplicate rows: {raw.duplicated().sum()}")
print(f"Missing values: {raw.isnull().sum()}")
print()
raw.head()

Shape: (790, 16)
Duplicate rows: 0
Missing values: competition           0
match_stage           0
date                  0
venue                 0
club                  0
opponent              0
match_score           0
player_position       0
goal_minute           0
score_at_goal         0
goal_type             0
assist_player         0
season                0
goal_decade           0
is_home_goal          0
goal_minute_bucket    0
dtype: int64



,competition,match_stage,date,venue,club,opponent,match_score,player_position,goal_minute,score_at_goal,goal_type,assist_player,season,goal_decade,is_home_goal,goal_minute_bucket
0,2ª B - Grupo III,2,9/5/2004,Home,FC Barcelona B,Girona FC,1:02,LW,65,1:02,Left-footed shot,Dani Fernández,2004-05,2000s,True,61-75
1,2ª B - Grupo III,4,9/18/2004,Home,FC Barcelona B,RCD Espanyol B,2:00,LW,88,2:00,Left-footed shot,Dani Fernández,2004-05,2000s,True,76-90
2,2ª B - Grupo III,10,10/31/2004,Away,FC Barcelona B,Villajoyosa CF,2:01,AM,2,0:01,Not reported,Not Applicable,2004-05,2000s,False,0-15
3,2ª B - Grupo III,14,11/28/2004,Away,FC Barcelona B,UE Figueres,2:02,AM,20,0:01,Not reported,Not Applicable,2004-05,2000s,False,16-30
4,2ª B - Grupo III,17,12/19/2004,Home,FC Barcelona B,CD Alcoyano,2:00,CF,40,2:00,Left-footed shot,Not Applicable,2004-05,2000s,True,31-45


No nulls and no duplicates, so the problems are in how values are written, not in what is missing. The four issues below would break an analysis.

## 2. Issues that need cleaning

### 2.1 `match_stage` mixes two kinds of values

Cup rounds are names ("Group Stage", "Final"). League games are matchday numbers stored as text ("1" to "38").

In [3]:
is_number = raw["match_stage"].str.isdigit()
print(f"Numeric matchdays: {is_number.sum()} rows")
print(f"Round names:       {(~is_number).sum()} rows")
print()
print("Round names:")
print(raw.loc[~is_number, "match_stage"].value_counts().to_string())

Numeric matchdays: 555 rows
Round names:       235 rows

Round names:
match_stage
Group Stage       84
Round of 16       56
Final             33
Quarter-Finals    27
Semi-Finals       22
First Round       10
Fourth Round       3


### 2.2 Scores are formatted like clock times

`match_score` and `score_at_goal` look like `1:02` and `4:00`. The part after the colon is zero padded, which suggests a spreadsheet read the score as a time (h:mm). They must be parsed into two numbers, and we need to decide which side is home.

In [4]:
print(raw[["match_score", "score_at_goal"]].describe().loc[["count", "unique", "top"]])
print()
raw[["date", "venue", "opponent", "match_score", "score_at_goal"]].sample(5, random_state=1)

       match_score score_at_goal
count          790           790
unique          49            39
top           3:01          1:00



,date,venue,opponent,match_score,score_at_goal
579,1/6/2019,Away,Getafe CF,1:02,0:01
148,11/7/2010,Away,Getafe CF,1:03,0:01
247,4/14/2012,Away,Levante UD,1:02,1:01
435,1/9/2016,Home,Granada CF,4:00,3:00
60,11/26/2008,Away,Sporting CP,2:05,0:03


### 2.3 `Not Applicable` in `assist_player`

This means the goal had no assist (penalties, free kicks, solo runs). It is different from missing data, so it should become an explicit flag.

In [5]:
no_assist = raw["assist_player"] == "Not Applicable"
print(f"No-assist rows: {no_assist.sum()} of {len(raw)} ({no_assist.mean():.1%})")
print()
print("Goal types among no-assist rows:")
print(raw.loc[no_assist, "goal_type"].value_counts().to_string())

No-assist rows: 240 of 790 (30.4%)

Goal types among no-assist rows:
goal_type
Penalty                   88
Direct free kick          59
Left-footed shot          58
Right-footed shot         15
Tap-in                     8
Header                     3
Solo run                   3
Not reported               2
Penalty rebound            2
Deflected shot on goal     1
Chest                      1


### 2.4 Too many `goal_type` groups

Several types have only a handful of rows, which makes charts unreadable. `Not reported` is genuinely missing.

In [6]:
print(f"Distinct raw goal types: {raw['goal_type'].nunique()}")
raw["goal_type"].value_counts().to_frame("rows")

Distinct raw goal types: 13


,rows
goal_type,
Left-footed shot,493
Right-footed shot,93
Penalty,88
Direct free kick,59
Header,28
Tap-in,10
Not reported,5
Solo run,5
Penalty rebound,3


### 2.5 `date` is text

Dates are written as month/day/year text (for example `9/5/2004`) and need to become real dates.

In [7]:
dates = pd.to_datetime(raw["date"], format="%m/%d/%Y")
print(f"Raw dtype: {raw['date'].dtype}")
print(f"Date range: {dates.min().date()} to {dates.max().date()}")
raw["date"].head(5).tolist()

Raw dtype: object
Date range: 2004-09-05 to 2026-03-07


['9/5/2004', '9/18/2004', '10/31/2004', '11/28/2004', '12/19/2004']

## 3. Other observations

These do not need cleaning but affect how results should be read.

In [8]:
print("Clubs:")
print(raw["club"].value_counts().to_string())
print()
print("Competitions:")
print(raw["competition"].value_counts().to_string())

Clubs:
club
FC Barcelona           672
Inter Miami CF          80
Paris Saint-Germain     32
FC Barcelona B           6

Competitions:
competition
LaLiga                    474
UEFA Champions League     129
Copa del Rey               56
MLS                        53
Ligue 1                    22
Supercopa                  14
Leagues Cup                12
MLS Cup Playoffs            7
CONCACAF Champions Cup      7
2ª B - Grupo III            6
Club World Cup              6
UEFA Super Cup              3
Trophée des Champions       1


In [9]:
minutes = raw["goal_minute"].astype(int)
print(f"Minute range: {minutes.min()} to {minutes.max()}")
print()
print("Goals by minute bucket (the 91+ bucket cannot separate stoppage time from extra time):")
print(raw["goal_minute_bucket"].value_counts().to_string())

Minute range: 2 to 110

Goals by minute bucket (the 91+ bucket cannot separate stoppage time from extra time):
goal_minute_bucket
76-90    161
61-75    136
16-30    134
31-45    132
46-60    120
0-15      74
91+       33


In [10]:
# The two home/away columns should say the same thing.
venue_is_home = raw["venue"] == "Home"
flag_is_home = raw["is_home_goal"].str.upper() == "TRUE"
print(f"Rows where venue and is_home_goal disagree: {(venue_is_home != flag_is_home).sum()}")

Rows where venue and is_home_goal disagree: 0


Notes for the report:
- The data covers **club goals only** (Barcelona, PSG, Inter Miami). There are no national team goals.
- Only goals are recorded. There are no shots, minutes played or positions held, so rates per game or per 90 minutes cannot be computed from this file.
- `is_home_goal` repeats `venue`, so one of them can be dropped.

## 4. Cleaning to-do list

This table is the input for `src/clean.py`. Notebook `01_cleaning_validation` checks that each item was handled correctly.

| # | Issue | Cleaning rule | Function in `clean.py` |
|---|---|---|---|
| 1 | `date` is text | Parse as month/day/year, add `year` and `month` | `clean_dates` |
| 2 | `match_stage` mixes names and numbers | Split into `stage_name` (League for numbered games) and `matchday` | `clean_stage` |
| 3 | Scores look like clock times | Parse as home:away, then convert to Messi's team point of view using `venue`; add score before the goal and game state | `clean_scores` |
| 4 | `Not Applicable` assists | Add `has_assist`; leave `assist_player` blank when there is no assist | `clean_assists` |
| 5 | 13 goal types | Group into 7; keep the raw value in `goal_type_raw` | `clean_goal_types` |
| 6 | Text booleans and minute bucket order | Convert `is_home_goal` to bool; make the minute bucket an ordered category | `clean_flags_and_categories` |
| 7 | No match identifier | Build `match_id` from date, club and opponent; add `goal_number_in_match` | `add_match_id` |